# Project 04 — Shared battery + cabin cooling

A reduced-order EV thermal-management study for a hot-ambient case where the cabin evaporator and battery chiller share finite refrigeration capacity.

**All numbers are illustrative and generic. No employer/product data is used.**


# Level 0 — Energy balance
## Learning journey
Start with cooling load, compressor work and condenser rejection before adding refrigerant-cycle detail.

$$Q_{total}=Q_{cabin}+Q_{battery}$$
$$COP=\\frac{Q_{cooling}}{W_{compressor}}$$
$$Q_{cond}=Q_{cooling}+W_{compressor}$$


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / 'model.py').exists():
    HERE = Path('projects/04-shared-battery-cabin-cooling')
sys.path.insert(0, str(HERE.resolve()))

from model import CoolingCase, energy_balance, allocate_capacity, ambient_sweep, transient_scenario

case = CoolingCase()
balance = energy_balance(case)
balance


For the default case, total requested cooling is 28 kW. At COP 2.5, compressor electrical input is 11.2 kW and condenser rejection is 39.2 kW.

The key distinction is that **cooling load, compressor electrical power and condenser heat rejection are three different quantities**.


In [ ]:
allocation = allocate_capacity(case.cabin_kw, case.battery_kw, case.available_cooling_kw, 'battery_priority')
allocation


# Level 1 — Ambient sensitivity and capacity allocation
The following trend is deliberately illustrative. It is not a compressor map. It is used only to make the shrinking hot-ambient margin visible.


In [ ]:
ambients = np.arange(25, 51, 1)
rows = ambient_sweep(ambients, cabin_kw=20, battery_kw=8, strategy='battery_priority')

T = np.array([r['ambient_c'] for r in rows])
capacity = np.array([r['available_capacity_kw'] for r in rows])
demand = np.array([r['demand_kw'] for r in rows])
cop = np.array([r['cop'] for r in rows])

plt.figure(figsize=(8, 4.5))
plt.plot(T, capacity, label='Illustrative available cooling capacity')
plt.plot(T, demand, label='Combined demand')
plt.xlabel('Ambient temperature [°C]')
plt.ylabel('Cooling [kW]')
plt.title('Hot-ambient refrigeration margin')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(T, cop)
plt.xlabel('Ambient temperature [°C]')
plt.ylabel('Illustrative COP [-]')
plt.title('Illustrative COP degradation with ambient')
plt.grid(True, alpha=0.3)
plt.show()


Interpretation: once the available refrigeration capacity drops below the combined cabin + battery demand, the control system cannot satisfy both loads simultaneously. Control can **allocate** limited capacity, but it cannot create missing hardware capacity.


# Level 2 preview — Transient stationary fast charge
Synthetic case: cabin pull-down demand starts high and decays, while battery cooling demand rises during the high-current fast-charge period.


In [ ]:
minutes = np.arange(0, 61, 1)
rows_t = transient_scenario(minutes, ambient_c=45, strategy='battery_priority')

m = np.array([r['minute'] for r in rows_t])
cab_d = np.array([r['cabin_demand_kw'] for r in rows_t])
bat_d = np.array([r['battery_demand_kw'] for r in rows_t])
cab_del = np.array([r['cabin_delivered_kw'] for r in rows_t])
bat_del = np.array([r['battery_delivered_kw'] for r in rows_t])
cap = np.array([r['available_capacity_kw'] for r in rows_t])

plt.figure(figsize=(9, 5))
plt.plot(m, cab_d, label='Cabin demand')
plt.plot(m, bat_d, label='Battery demand')
plt.plot(m, cab_del, '--', label='Cabin delivered')
plt.plot(m, bat_del, '--', label='Battery delivered')
plt.plot(m, cap, ':', label='Available shared capacity')
plt.xlabel('Time [min]')
plt.ylabel('Cooling [kW]')
plt.title('Shared cooling during stationary fast charge')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## Engineering interpretation

- Battery-priority control protects battery cooling first when the shared system saturates.
- The price is unmet cabin cooling demand during the strongest combined-load period.
- If both cabin and battery temperatures rise even when compressor command is near maximum, compressor speed alone is not enough to diagnose the root cause.
- The next fidelity layer must inspect refrigerant-side state and heat-rejection capability.


# Next fidelity gate
Before calling this a refrigeration model, add:

1. compressor map/envelope,
2. condensing and evaporating temperature,
3. high-/low-side saturation states,
4. superheat and subcooling,
5. condenser approach,
6. battery heat-generation model,
7. cabin sensible/latent load,
8. validation against a published dataset.

**Trust boundary:** the current model is suitable for energy-balance and capacity-allocation reasoning, not refrigerant fault diagnosis.
